# Machine Learning Systems for Sleep Quality Assessment Based on EEG Signals

---

* **Author**: Carmen-Theodora Craciun
* **The purpose of this Notebook**: This notebook handles ...
* **Datasets**: [Sleep-EDFX Database](https://www.physionet.org/content/sleep-edfx/1.0.0/) and [University College Dublin Sleep Apnea Database](https://physionet.org/content/ucddb/1.0.0/)
    * Sleep Cassette (SC) - the study on healthy people;
    * Sleep Telemetry (ST) - sleep study in people with difficulty falling asleep. This study was designed to look at the effects of temazepam, a drug with hypnotic effects;
    * UCDDB - patients diagnosed with apnea, who do not suffer from cardiological diseases or autonomic dysfunctions and are not taking medication at the time of registration.
* **Input**:
  * class_distribution.csv
  * results.zip - a compressed archive containing the layered and prepared dataset for training (train, test, val)
    * each file in these folders represents a *processed sleep window* (epoch)
* **Output:**
  * `model_bilstm_baseline.pt`
  * `model_crnn.pt`
  * `model_resnet.pt`

# Imports

In [2]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
import numpy as np
import joblib
from sklearn.base import clone
from tensorflow.keras import layers, models
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
import gc
import shutil
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization, Dense, Dropout, Input, GlobalAveragePooling1D
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from scipy.ndimage import gaussian_filter1d

In [4]:
train_path_csv = 'process_dataset/extracted_features/train_features.csv'
val_path_csv = 'process_dataset/extracted_features/val_features.csv'

train_path_raw = 'process_dataset/train'
val_path_raw = 'process_dataset/val'

class_distribution_path = 'process_dataset/class_distribution_train.csv'

In [5]:
import utils

# Cross Validation

## ML

In [ ]:
def cross_validation(
    model, X, y, subjects,
    model_name, model_type, save_path='./models',
    n_splits=5, sample_weights=None,
    to_calibrate=False
):
    '''Cross-validate for LightGBM, XGBoost, Random Forest, LDA and Logistic Regression.'''
    print(f"=== Cross Validation for {model_name} ({n_splits} folds) ===")

    X_arr = np.array(X)
    y_arr = np.array(y)
    subjects_arr = np.array(subjects)

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_accuracies = []
    fold_kappas = []

    best_acc = 0.0
    best_model = None

    if not os.path.exists(save_path):
        os.makedirs(save_path)

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_arr, y_arr, groups=subjects_arr), 1):
        print(f"===== FOLD {fold}/{n_splits} =====")
        
        X_train, X_val = X_arr[train_idx], X_arr[val_idx]
        y_train, y_val = y_arr[train_idx], y_arr[val_idx]

        cloned_model = clone(model)

        # Train base model
        if sample_weights is None:
            cloned_model.fit(X_train, y_train)
        else:
            cloned_model.fit(X_train, y_train, sample_weight=sample_weights[train_idx])

        # Optional calibration
        if to_calibrate:
            calibrated_model = CalibratedClassifierCV(
                estimator=cloned_model,
                method='sigmoid',
                cv=5
            )
            calibrated_model.fit(X_train, y_train)
            model_to_eval = calibrated_model
        else:
            model_to_eval = cloned_model

        # Predict on validation
        y_pred = model_to_eval.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        kappa = cohen_kappa_score(y_val, y_pred)

        fold_accuracies.append(acc)
        fold_kappas.append(kappa)

        msg = f"Accuracy = {acc:.4f}, Kappa = {kappa:.4f}"

        # Save best model
        if acc > best_acc:
            best_acc = acc
            best_model = model_to_eval
            msg += " -> Best model"

        print(msg)
        print()
            

    best_model_path = os.path.join(save_path, f"best_{model_name}.pkl")
    joblib.dump(best_model, best_model_path)

    print("\n===== FINAL RESULTS =====")
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    mean_kappa = np.mean(fold_kappas)
    std_kappa = np.std(fold_kappas)
    
    print(f"Mean ACC:   {mean_acc:.4f}")
    print(f"Mean KAPPA: {mean_kappa:.4f}")
    print(f"Best model saved at: {best_model_path}")

    # results_dict = {}
    
    results_dict = {
        "name": model_name,
        "type": model_type,
        "acc_mean": float(mean_acc),
        "acc_std": float(std_acc),
        "kappa_mean": float(mean_kappa),
        "kappa_std": float(std_kappa),
        "acc_best": float(max(fold_accuracies)),
        "kappa_best": float(max(fold_kappas)),
        "fold_accs": fold_accuracies,
        "fold_kappas": fold_kappas,
        "path": best_model_path
    }

    return results_dict

## DL on extracted

In [ ]:
def cross_validate_keras_model(
    X,y,subjects,build_model_fn,
    model_name,model_type,n_splits=5,
    batch_size=32,epochs=50,
    callbacks_list=None,
    verbose=1,save_path="best_models"
):
    '''Cross-validate for MLP, LSTM and ResNet.'''
    os.makedirs(save_path, exist_ok=True)

    X = np.array(X)
    y = np.array(y)
    subjects = np.array(subjects)

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_accuracies = []
    fold_kappas = []
    best_acc = -1
    best_kappa = 0
    best_model_path = None

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=subjects), 1):
        print(f"===== FOLD {fold}/{n_splits} =====")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Detect input shape
        input_shape = X_train.shape[1:]
        num_classes = len(np.unique(y))

        # Build model
        model = build_model_fn(input_shape, num_classes)

        # Callbacks
        callbacks = callbacks_list() if callbacks_list is not None else None

        # Train
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=verbose
        )

        # Predict
        y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)

        acc = accuracy_score(y_val, y_pred)
        kappa = cohen_kappa_score(y_val, y_pred)

        fold_accuracies.append(acc)
        fold_kappas.append(kappa)

        msg = f"Accuracy = {acc:.4f}, Kappa = {kappa:.4f}"

        # Save best model
        if acc > best_acc:
            best_acc = acc
            best_kappa = kappa
            best_model_path = os.path.join(save_path, f"best_{model_name}.keras")
            model.save(best_model_path)
            msg += " -> Best model"

        print(msg)
        print()

        # Cleanup
        del model
        tf.keras.backend.clear_session()
        gc.collect()

    print("\n===== FINAL RESULTS =====")
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    mean_kappa = np.mean(fold_kappas)
    std_kappa = np.std(fold_kappas)
    
    print(f"Mean ACC:   {mean_acc:.4f}")
    print(f"Mean KAPPA: {mean_kappa:.4f}")
    print(f"Best model saved at: {best_model_path}")

    # results_dict = {}
    
    results_dict = {
        "name": model_name,
        "type": model_type,
        "acc_mean": float(mean_acc),
        "acc_std": float(std_acc),
        "kappa_mean": float(mean_kappa),
        "kappa_std": float(std_kappa),
        "acc_best": float(best_acc),
        "kappa_best": float(best_kappa),
        "fold_accs": fold_accuracies,
        "fold_kappas": fold_kappas,
        "path": best_model_path
    }

    return results_dict

## CNN

In [ ]:
def kfold_cnn_generators_grouped(
    X_paths,y,subjects,
    build_model_fn,
    model_name,model_type,
    generator_fn,batch_size,
    callbacks_fn = None,
    epochs=40,n_splits=5,
    save_path="best_models",
    verbose=1
):
    '''Cross-validate for CNN.'''
    os.makedirs(save_path, exist_ok=True)

    cv = GroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_accuracies = []
    fold_kappas = []
    best_acc = -1
    best_model_path = None

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_paths, y, groups=subjects), 1):
        print(f"===== FOLD {fold}/{n_splits} =====")

        fold_train_dir = f"fold_train_{fold}"
        fold_val_dir   = f"fold_val_{fold}"

        os.makedirs(fold_train_dir, exist_ok=True)
        os.makedirs(fold_val_dir, exist_ok=True)

        for idx in train_idx:
            shutil.copy(X_paths[idx], fold_train_dir)

        for idx in val_idx:
            shutil.copy(X_paths[idx], fold_val_dir)

        train_gen = generator_fn(fold_train_dir, True, batch_size)
        val_gen   = generator_fn(fold_val_dir, False, batch_size)

        # Model
        sample_batch, _ = train_gen[0]
        input_shape = sample_batch.shape[1:]

        model = build_model_fn(input_shape=input_shape)

        if callbacks_fn is None:
            callbacks = []
        else: callbacks = callbacks_fn()

        # Train
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=callbacks,
            verbose=verbose
        )

        # Evaluate
        y_true = []
        y_pred = []

        for Xb, yb in val_gen:
            y_true.extend(np.argmax(yb, axis=1))
            y_pred.extend(np.argmax(model.predict(Xb, verbose=0), axis=1))

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        acc = accuracy_score(y_true, y_pred)
        kappa = cohen_kappa_score(y_true, y_pred)

        fold_accuracies.append(acc)
        fold_kappas.append(kappa)
                
        msg = f"Accuracy = {acc:.4f}, Kappa = {kappa:.4f}"

        # 7. Salvăm cel mai bun model
        if acc > best_acc:
            best_acc = acc
            best_model_path = os.path.join(save_path, f"best_{model_name}.keras")
            model.save(best_model_path)
            msg += " -> Best model"

        print(msg)
        print()

        # Clean the memory
        del model, train_gen, val_gen
        tf.keras.backend.clear_session()
        gc.collect()

        shutil.rmtree(fold_train_dir)
        shutil.rmtree(fold_val_dir)


    print("\n===== FINAL RESULTS =====")
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    mean_kappa = np.mean(fold_kappas)
    std_kappa = np.std(fold_kappas)
    
    print(f"Mean ACC:   {mean_acc:.4f}")
    print(f"Mean KAPPA: {mean_kappa:.4f}")
    print(f"Best model saved at: {best_model_path}")

    # results_dict = {}
    
    results_dict = {
        "name": model_name,
        "type": model_type,
        "acc_mean": float(mean_acc),
        "acc_std": float(std_acc),
        "kappa_mean": float(mean_kappa),
        "kappa_std": float(std_kappa),
        "acc_best": float(best_acc),
        "kappa_best": float(max(fold_kappas)),
        "fold_accs": fold_accuracies,
        "fold_kappas": fold_kappas,
        "path": best_model_path
    }

    return results_dict

## SeqSleepNet

In [6]:
def crossval_seqsleepnet(
    X_paths,y,subjects,
    build_model_fn,
    model_name,model_type,
    generator_fn,
    batch_size = 32,
    epochs=40,n_splits=5,
    save_path="best_models",
    verbose=1
):
    os.makedirs(save_path, exist_ok=True)

    cv = GroupKFold(n_splits=n_splits)

    fold_accuracies = []
    fold_kappas = []
    best_acc = -1
    best_model_path = None

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_paths, y, groups=subjects), 1):
        print(f"===== FOLD {fold}/{n_splits} =====")

        # Create fold directories
        fold_train = f"fold_train_{fold}"
        fold_val   = f"fold_val_{fold}"
        os.makedirs(fold_train, exist_ok=True)
        os.makedirs(fold_val, exist_ok=True)

        for idx in train_idx:
            shutil.copy(X_paths[idx], fold_train)
        for idx in val_idx:
            shutil.copy(X_paths[idx], fold_val)

        # Generators
        train_gen = generator_fn(fold_train, True, batch_size)
        val_gen   = generator_fn(fold_val, False, batch_size)

        # Build + compile model
        model = build_model_fn()
        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-3),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
            metrics=["accuracy"]
        )

        es = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=7,
            restore_best_weights=True,
            verbose=1
        )

        lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )

        # Train
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=[es, lr],
            verbose=verbose
        )

        # True labels
        y_true = []
        for _, yb in val_gen:
            y_true.extend(np.argmax(yb, axis=1))
        y_true = np.array(y_true)

        # Raw predictions
        y_pred_proba = model.predict(val_gen, verbose=0)

        # === PIPELINE FINAL ===
        sigma_list = [0.5, 1.0, 1.5]
        probas_smooth = [
            gaussian_filter1d(y_pred_proba, sigma=s, axis=0, mode="nearest")
            for s in sigma_list
        ]
        proba_smooth = np.mean(probas_smooth, axis=0)

        temps = [0.85, 0.45, 0.75, 0.95, 0.75]
        A = utils.compute_transition_matrix_adaptive(y_true, temps)
        y_hmm = utils.viterbi(proba_smooth, A)

        y_med = utils.adaptive_median(y_hmm)
        y_final = utils.physiologic_rules(y_med)

        # Metrics
        acc = np.mean(y_final == y_true)
        kappa = cohen_kappa_score(y_true, y_final)

        fold_accuracies.append(acc)
        fold_kappas.append(kappa)

        msg = f"Fold ACC={acc:.4f}, Kappa={kappa:.4f}"

        if acc > best_acc:
            best_acc = acc
            best_model_path = os.path.join(save_path, f"best_{model_name}.keras")
            model.save(best_model_path)
            msg += "  <-- BEST"

        print(msg)

        # Cleanup
        del model, train_gen, val_gen
        tf.keras.backend.clear_session()
        gc.collect()
        shutil.rmtree(fold_train)
        shutil.rmtree(fold_val)
        print()

    print("\n===== FINAL RESULTS =====")
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    mean_kappa = np.mean(fold_kappas)
    std_kappa = np.std(fold_kappas)
    
    print(f"Mean ACC:   {mean_acc:.4f}")
    print(f"Mean KAPPA: {mean_kappa:.4f}")
    print(f"Best model saved at: {best_model_path}")

    return {
        "name": model_name,
        "type": model_type,
        "acc_mean": float(mean_acc),
        "acc_std": float(std_acc),
        "kappa_mean": float(mean_kappa),
        "kappa_std": float(std_kappa),
        "acc_best": float(best_acc),
        "kappa_best": float(max(fold_kappas)),
        "accs": fold_accuracies,
        "kappas": fold_kappas,
        "file_name": f"best_{model_name}.keras"
    }

# Dataset

In [11]:
import pandas as pd

df_train = pd.read_csv(train_path_csv)
df_val = pd.read_csv(val_path_csv)

df_total = pd.concat([df_train, df_val], axis=0)

subjects = df_total['Subject_ID'].values

cols_to_drop = ['Subject_ID', 'Dataset_Source', 'Label']
feature_cols = [c for c in df_total.columns if c not in cols_to_drop]

X_total = df_total[feature_cols]
y_total = df_total['Label']

scaler = StandardScaler()
X_total_scaled = scaler.fit_transform(X_total.fillna(0))

X_total_seq, y_total_seq, subjects_seq = utils.build_sequences_per_subject(df_total, feature_cols, seq_len=30, step=5)

class_distribution = pd.read_csv(class_distribution_path)

class_weights = dict(zip(class_distribution['Class_ID'], class_distribution['Weight']))

sample_weights_custom = np.array([class_weights[y] for y in y_total])

In [12]:
def get_file_list_from_folders(*folders):
    file_list = []
    for folder in folders:
        for f in os.listdir(folder):
            if f.endswith(".npz"):
                file_list.append(os.path.join(folder, f))
    return np.array(file_list)

def create_generator(directory, is_training, batch_size):
    return utils.SleepDataGenerator(
        directory=directory,
        is_training=is_training,
        batch_size=batch_size,
        n_classes=5,
        use_context=True,
        window_size=3,
        use_lag_lead=True,
        downsample=True,
        target_length=1500,
        sampling_rate=50,
        lag_seconds=0.5
    )

def extract_subjects(file_list):
    subjects = []
    for f in file_list:
        fname = os.path.basename(f)
        digits = ''.join([c for c in fname if c.isdigit()])
        subjects.append(int(digits[:2]))
    return np.array(subjects)

def extract_labels(file_list):
    labels = []
    for f in file_list:
        data = np.load(f, allow_pickle=True)

        y_raw = data["y"] if "y" in data else data["arr_1"]
        y_raw = np.array(y_raw).reshape(-1)
        label = int(np.bincount(y_raw).argmax())

        labels.append(label)

    return np.array(labels)

train_path = "process_dataset/train"
val_path   = "process_dataset/val"

file_list = get_file_list_from_folders(train_path, val_path)
labels = extract_labels(file_list)
subjects = extract_subjects(file_list)

# Models Cross Validation

In [13]:
results = []

## LightGBM

In [53]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.08,
    max_depth=10,
    num_leaves=125,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)


result = cross_validation(
    model=lgb_model,
    X=X_total,
    y=y_total,
    subjects=subjects,
    model_name="LGBM_calibrated",
    model_type='Tree-Based',
    sample_weights=sample_weights_custom,
    save_path="./models",
    to_calibrate=True
)


results.append(result)

=== Cross Validation for LGBM_calibrated (5 folds) ===
===== FOLD 1/5 =====
Accuracy = 0.7822, Kappa = 0.7029 -> Best model

===== FOLD 2/5 =====
Accuracy = 0.7666, Kappa = 0.6816

===== FOLD 3/5 =====
Accuracy = 0.7568, Kappa = 0.6683

===== FOLD 4/5 =====
Accuracy = 0.7642, Kappa = 0.6797

===== FOLD 5/5 =====
Accuracy = 0.7858, Kappa = 0.7113 -> Best model


===== FINAL RESULTS =====
Mean ACC:   0.7711
Mean KAPPA: 0.6888
Best model saved at: ./models/best_LGBM_calibrated.pkl


## XGBoost

In [79]:
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    random_state=42,
    n_estimators=400,
    learning_rate=0.08,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    enable_categorical=True,
    n_jobs=-1
)

result = cross_validation(
    model=xgb_model,
    X=X_total,
    y=y_total,
    subjects=subjects,
    model_name="XGB_calibrated",
    model_type='Tree-Based',
    sample_weights=sample_weights_custom,
    save_path="./models",
    to_calibrate=True
)

results.append(result)

=== Cross Validation for XGB_calibrated (5 folds) ===
===== FOLD 1/5 =====
Accuracy = 0.7802, Kappa = 0.7003 -> Best model

===== FOLD 2/5 =====
Accuracy = 0.7679, Kappa = 0.6838

===== FOLD 3/5 =====
Accuracy = 0.7565, Kappa = 0.6681

===== FOLD 4/5 =====
Accuracy = 0.7636, Kappa = 0.6793

===== FOLD 5/5 =====
Accuracy = 0.7836, Kappa = 0.7086 -> Best model


===== FINAL RESULTS =====
Mean ACC:   0.7704
Mean KAPPA: 0.6880
Best model saved at: ./models/best_XGB_calibrated.pkl


## Random Forest

In [81]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

result = cross_validation(
    model=rf_model,
    X=X_total,
    y=y_total,
    subjects=subjects,
    model_name="RF_calibrated",
    model_type='Tree-Based',
    sample_weights=sample_weights_custom,
    save_path="./models",
    to_calibrate=True
)

results.append(result)

=== Cross Validation for RF_calibrated (5 folds) ===
===== FOLD 1/5 =====
Accuracy = 0.7524, Kappa = 0.6625 -> Best model

===== FOLD 2/5 =====
Accuracy = 0.7389, Kappa = 0.6433

===== FOLD 3/5 =====
Accuracy = 0.7308, Kappa = 0.6330

===== FOLD 4/5 =====
Accuracy = 0.7340, Kappa = 0.6386

===== FOLD 5/5 =====
Accuracy = 0.7602, Kappa = 0.6763 -> Best model


===== FINAL RESULTS =====
Mean ACC:   0.7433
Mean KAPPA: 0.6507
Best model saved at: ./models/best_RF_calibrated.pkl


## LDA

In [83]:
lda_lsqr = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')

result = cross_validation(
    model=lda_lsqr,
    X=X_total_scaled,
    y=y_total,
    subjects=subjects,
    model_name="LDA-Baseline",
    model_type='Probabilistic',
    save_path="./models"
)

results.append(result)

=== Cross Validation for LDA-Baseline (5 folds) ===
===== FOLD 1/5 =====
Accuracy = 0.7391, Kappa = 0.6459 -> Best model

===== FOLD 2/5 =====
Accuracy = 0.7234, Kappa = 0.6239

===== FOLD 3/5 =====
Accuracy = 0.7122, Kappa = 0.6098

===== FOLD 4/5 =====
Accuracy = 0.7134, Kappa = 0.6116

===== FOLD 5/5 =====
Accuracy = 0.7377, Kappa = 0.6467


===== FINAL RESULTS =====
Mean ACC:   0.7251
Mean KAPPA: 0.6276
Best model saved at: ./models/best_LDA-Baseline.pkl


## Logistic Regression

In [85]:
lr_lbfgs = LogisticRegression(
    class_weight='balanced',
    max_iter=5000,
    C=0.5,
    solver='lbfgs',
    random_state=42
)

result = cross_validation(
    model=lr_lbfgs,
    X=X_total_scaled,
    y=y_total,
    subjects=subjects,
    model_name="LogisticR-Baseline",
    model_type='Probabilistic',
    save_path="./models"
)

results.append(result)

=== Cross Validation for LogisticR-Baseline (5 folds) ===
===== FOLD 1/5 =====
Accuracy = 0.7299, Kappa = 0.6477 -> Best model

===== FOLD 2/5 =====
Accuracy = 0.7033, Kappa = 0.6137

===== FOLD 3/5 =====
Accuracy = 0.6939, Kappa = 0.6031

===== FOLD 4/5 =====
Accuracy = 0.7116, Kappa = 0.6239

===== FOLD 5/5 =====
Accuracy = 0.7218, Kappa = 0.6407


===== FINAL RESULTS =====
Mean ACC:   0.7121
Mean KAPPA: 0.6258
Best model saved at: ./models/best_LogisticR-Baseline.pkl


## MLP

In [87]:
def build_mlp(input_shape, n_classes, dropout_rate=0.3):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation='relu'),
        layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [90]:
result = cross_validate_keras_model(
    X=X_total,
    y=y_total,
    subjects=subjects,
    build_model_fn=build_mlp,
    model_name="MLP",
    model_type='ANN',
    batch_size=32,
    epochs=50,
    callbacks_list=utils.make_callbacks,
    save_path="./models"
)

results.append(result)

===== FOLD 1/5 =====
Epoch 1/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 33s 4ms/step - accuracy: 0.7161 - loss: 0.7450 - val_accuracy: 0.7571 - val_loss: 0.6423 - learning_rate: 0.0010
Epoch 2/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 30s 4ms/step - accuracy: 0.7453 - loss: 0.6670 - val_accuracy: 0.7686 - val_loss: 0.6150 - learning_rate: 0.0010
Epoch 3/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - accuracy: 0.7534 - loss: 0.6424 - val_accuracy: 0.7716 - val_loss: 0.6004 - learning_rate: 0.0010
Epoch 4/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.7601 - loss: 0.6247 - val_accuracy: 0.7737 - val_loss: 0.5943 - learning_rate: 0.0010
Epoch 5/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - accuracy: 0.7661 - loss: 0.6112 - val_accuracy: 0.7742 - val_loss: 0.5902 - learning_rate: 0.0010
Epoch 6/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.7677 - loss: 0.6013 - val_accuracy: 0.7759 - val_loss: 0.5863 - learning_rate: 0.0010
Epoch 7/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 30s 4ms

## LSTM

In [ ]:
def build_lstm_se_focal(input_shape, n_classes, units=128, dropout=0.4, recurrent_dropout=0.2):
    inputs = layers.Input(shape=input_shape)

    # LSTM 1 (return sequences)
    x = layers.LSTM(
        units,
        return_sequences=True,
        dropout=dropout,
        recurrent_dropout=recurrent_dropout
    )(inputs)
    x = layers.BatchNormalization()(x)

    # LSTM 2 (return last hidden state)
    x = layers.LSTM(
        units // 2,
        dropout=dropout,
        recurrent_dropout=recurrent_dropout
    )(x)
    x = layers.BatchNormalization()(x)

    # SE block
    x = utils.se_feature_attention(x)

    # Dense layers
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer='l2')(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer='adam',
        loss=utils.categorical_focal_loss(gamma=2.0, alpha=0.25),
        metrics=['accuracy']
    )
    return model

In [93]:
result = cross_validate_keras_model(
    X=X_total_seq,
    y=y_total_seq,
    subjects=subjects_seq,
    build_model_fn=build_lstm_se_focal,
    model_name="lstm_se_focal",
    model_type='ANN',
    batch_size=32,
    epochs=50,
    callbacks_list=utils.make_callbacks,
    save_path="./models"
)

results.append(result)

===== FOLD 1/5 =====
Epoch 1/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 90s 61ms/step - accuracy: 0.6512 - loss: 0.1743 - val_accuracy: 0.7219 - val_loss: 0.0947 - learning_rate: 0.0010
Epoch 2/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 88s 64ms/step - accuracy: 0.7156 - loss: 0.0952 - val_accuracy: 0.7297 - val_loss: 0.0888 - learning_rate: 0.0010
Epoch 3/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 86s 63ms/step - accuracy: 0.7354 - loss: 0.0858 - val_accuracy: 0.7456 - val_loss: 0.0840 - learning_rate: 0.0010
Epoch 4/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 89s 65ms/step - accuracy: 0.7463 - loss: 0.0818 - val_accuracy: 0.7307 - val_loss: 0.0885 - learning_rate: 0.0010
Epoch 5/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 89s 65ms/step - accuracy: 0.7517 - loss: 0.0784 - val_accuracy: 0.7509 - val_loss: 0.0823 - learning_rate: 0.0010
Epoch 6/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 89s 65ms/step - accuracy: 0.7567 - loss: 0.0758 - val_accuracy: 0.7492 - val_loss: 0.0815 - learning_rate: 0.0010
Epoch 7/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 8

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1366/1366 ━━━━━━━━━━━━━━━━━━━━ 76s 55ms/step - accuracy: 0.7914 - loss: 0.0575 - val_accuracy: 0.7710 - val_loss: 0.0714 - learning_rate: 5.0000e-04
Epoch 20/50
1366/1366 ━━━━━━━━━━━━━━━━━━━━ 78s 57ms/step - accuracy: 0.7919 - loss: 0.0566 - val_accuracy: 0.7747 - val_loss: 0.0721 - learning_rate: 5.0000e-04
Epoch 21/50
1366/1366 ━━━━━━━━━━━━━━━━━━━━ 78s 57ms/step - accuracy: 0.7923 - loss: 0.0560 - val_accuracy: 0.7748 - val_loss: 0.0724 - learning_rate: 5.0000e-04
Epoch 22/50
1366/1366 ━━━━━━━━━━━━━━━━━━━━ 76s 56ms/step - accuracy: 0.7921 - loss: 0.0568 - val_accuracy: 0.7759 - val_loss: 0.0725 - learning_rate: 5.0000e-04
Epoch 23/50
1366/1366 ━━━━━━━━━━━━━━━━━━━━ 74s 54ms/step - accuracy: 0.7941 - loss: 0.0555 - val_accuracy: 0.7763 - val_loss: 0.0725 - learning_rate: 5.0000e-04
Epoch 24/50
1366/1366 ━━━━━━━━━━━━━━━━━━━━ 77s 56ms/step - accuracy: 0.8018 - loss: 0.0528 - val_accuracy: 0.7758 - val_loss: 0.0720 - learning_rate: 2.5000e-04
Epoch 25/50
1366/1366 ━━━━━━━━━━━━━━━━━━━━ 78s

## ResNet

In [18]:
def build_resnet_feature_model(seq_len, n_features, n_classes=5):

    inp = layers.Input(shape=(seq_len, n_features))

    def conv_block(x, filters):
        shortcut = x

        x = layers.Conv1D(filters, 3, padding='same',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        x = layers.Conv1D(filters, 3, padding='same',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)

        if shortcut.shape[-1] != filters:
            shortcut = layers.Conv1D(filters, 1, padding='same',
                                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(shortcut)

        x = layers.Add()([x, shortcut])
        x = layers.ReLU()(x)
        x = layers.Dropout(0.2)(x)
        return x

    x = conv_block(inp, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model

def build_resnet_wrapper(input_shape,num_classes=5):
    seq_len = input_shape[0]
    n_features = input_shape[1]
    return build_resnet_feature_model(seq_len, n_features, num_classes)

In [19]:
result = cross_validate_keras_model(
    X=X_total_seq,
    y=y_total_seq,
    subjects=subjects_seq,
    build_model_fn=build_resnet_wrapper,
    model_name="resnet_feature",
    model_type='ANN',
    batch_size=32,
    epochs=50,
    callbacks_list=utils.make_callbacks,
    save_path="./models"
)

results.append(result)

===== FOLD 1/5 =====


E0000 00:00:1779824276.561925  538478 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.6249 - loss: 1.0441 - val_accuracy: 0.6964 - val_loss: 0.8600 - learning_rate: 3.0000e-04
Epoch 2/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 34s 25ms/step - accuracy: 0.7156 - loss: 0.8177 - val_accuracy: 0.7426 - val_loss: 0.7724 - learning_rate: 3.0000e-04
Epoch 3/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 34s 25ms/step - accuracy: 0.7475 - loss: 0.7372 - val_accuracy: 0.7457 - val_loss: 0.7546 - learning_rate: 3.0000e-04
Epoch 4/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 34s 25ms/step - accuracy: 0.7628 - loss: 0.6915 - val_accuracy: 0.7470 - val_loss: 0.7242 - learning_rate: 3.0000e-04
Epoch 5/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 36s 26ms/step - accuracy: 0.7731 - loss: 0.6702 - val_accuracy: 0.7539 - val_loss: 0.7267 - learning_rate: 3.0000e-04
Epoch 6/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 35s 26ms/step - accuracy: 0.7796 - loss: 0.6467 - val_accuracy: 0.7487 - val_loss: 0.7268 - learning_rate: 3.0000e-04
Epoch 7/50
1365/1365 ━━━━━━━━━━━━━━━━━━━

## 1D-CNN

In [ ]:
# def get_file_list_from_folders(*folders):
#     file_list = []
#     for folder in folders:
#         for f in os.listdir(folder):
#             if f.endswith(".npz"):
#                 file_list.append(os.path.join(folder, f))
#     return np.array(file_list)

# def create_generator(directory, is_training, batch_size):
#     return utils.SleepDataGenerator(
#         directory=directory,
#         is_training=is_training,
#         batch_size=batch_size,
#         n_classes=5,
#         use_context=True,
#         window_size=3,
#         use_lag_lead=True,
#         downsample=True,
#         target_length=1500,
#         sampling_rate=50,
#         lag_seconds=0.5
#     )

# def extract_subjects(file_list):
#     subjects = []
#     for f in file_list:
#         fname = os.path.basename(f)
#         digits = ''.join([c for c in fname if c.isdigit()])
#         subjects.append(int(digits[:2]))   # primele 2 cifre = ID subiect
#     return np.array(subjects)

# def extract_labels(file_list):
#     labels = []
#     for f in file_list:
#         data = np.load(f, allow_pickle=True)

#         # y_raw este (n_epochs,)
#         y_raw = data["y"] if "y" in data else data["arr_1"]

#         # îl forțăm să fie vector 1D
#         y_raw = np.array(y_raw).reshape(-1)

#         # eticheta fișierului = cea mai frecventă etichetă
#         label = int(np.bincount(y_raw).argmax())

#         labels.append(label)

#     return np.array(labels)

# def build_medium_cnn(input_shape=(9000, 2)):
#     model = Sequential([
#         Input(shape=input_shape),
#         Conv1D(32, 50, strides=4, activation='relu', padding='same'),
#         MaxPooling1D(4),
#         Conv1D(64, 15, strides=2, activation='relu', padding='same'),
#         MaxPooling1D(4),
#         Conv1D(128, 7, activation='relu', padding='same'),
#         MaxPooling1D(2),
#         Conv1D(256, 5, activation='relu', padding='same'),
#         GlobalAveragePooling1D(),
#         Dense(128, activation='relu'),
#         Dropout(0.5),
#         Dense(5, activation='softmax')
#     ])
#     model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
#     return model

# train_path = "process_dataset/train"
# val_path   = "process_dataset/val"

# file_list = get_file_list_from_folders(train_path, val_path)
# labels = extract_labels(file_list)
# subjects = extract_subjects(file_list)

In [10]:
from tensorflow.keras.optimizers import Adam


result = kfold_cnn_generators_grouped(
    X_paths=file_list,
    y=labels,
    subjects=subjects,
    build_model_fn=build_medium_cnn,
    model_name="medium_cnn",
    model_type='ANN-Raw',
    generator_fn=create_generator,
    callbacks_fn=utils.make_callbacks,
    batch_size=32,
    epochs=50,
    save_path="./models"
)

results.append(result)

===== FOLD 1/5 =====


E0000 00:00:1779948656.130974  648648 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 224s 910ms/step - accuracy: 0.3774 - loss: 1.4942 - val_accuracy: 0.4899 - val_loss: 1.2541 - learning_rate: 0.0010
Epoch 2/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 223s 909ms/step - accuracy: 0.5074 - loss: 1.2455 - val_accuracy: 0.5117 - val_loss: 1.2156 - learning_rate: 0.0010
Epoch 3/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 215s 880ms/step - accuracy: 0.5503 - loss: 1.1239 - val_accuracy: 0.5755 - val_loss: 1.0615 - learning_rate: 0.0010
Epoch 4/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 226s 922ms/step - accuracy: 0.5907 - loss: 1.0291 - val_accuracy: 0.5875 - val_loss: 1.0183 - learning_rate: 0.0010
Epoch 5/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 224s 914ms/step - accuracy: 0.6139 - loss: 0.9805 - val_accuracy: 0.5858 - val_loss: 1.0525 - learning_rate: 0.0010
Epoch 6/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 221s 903ms/step - accuracy: 0.6219 - loss: 0.9701 - val_accuracy: 0.6339 - val_loss: 0.9943 - learning_rate: 0.0010
Epoch 7/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 225s 917ms/step - accura

## SeqSleepNet

In [ ]:
class Attention(layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, inputs):
        score = self.V(tf.nn.tanh(self.W(inputs)))
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights * inputs, axis=1)
        return context
        
def build_seqsleepnet(input_length=4500, frame_length=50, frame_units=64, epoch_units=128):
    num_frames = input_length // frame_length

    inputs = layers.Input(shape=(input_length, 1))

    # reshape into frames
    x = layers.Reshape((num_frames, frame_length, 1))(inputs)

    # frame-level encoding
    x = layers.TimeDistributed(layers.Bidirectional(layers.GRU(frame_units, return_sequences=True)))(x)
    x = layers.TimeDistributed(layers.Bidirectional(layers.GRU(frame_units)))(x)

    # epoch-level encoding
    x = layers.Bidirectional(layers.GRU(epoch_units, return_sequences=True))(x)

    # attention
    x = utils.Attention(epoch_units)(x)

    # classifier
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(5, activation="softmax")(x)

    return models.Model(inputs, outputs)

result = crossval_seqsleepnet(
    X_paths = file_list,y=labels,subjects=subjects,
    build_model_fn = lambda: build_seqsleepnet(),
    model_name = 'SeqSleepNet',model_type='Hybrid',
    generator_fn = lambda path, train, bs: utils.SleepDataGenerator(
        path,
        batch_size=bs,
        is_training=train,
        use_context=True,
        window_size=3,
        use_lag_lead=False,
        downsample=True,
        target_length=1500,
        sampling_rate=50,
        lag_seconds=0.5,
        select_channels=[0]
    ),batch_size = 32,
    epochs=50,
    save_path="./models",
)

results.append(result)

===== FOLD 1/5 =====
Epoch 1/50


In [ ]:
results

In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv("./model_seq_results.csv", index=False)

# Results

In [ ]:
df_results = pd.DataFrame(results)
df_results['name'] = df_results['name'].replace({
    'LGBM_calibrated': 'LGBM',
    'XGB_calibrated': 'XGB',
    'RF_calibrated': 'RF',
    'LDA-Baseline':'LDA',
    'LogisticR-Baseline':'LR',
    'lstm_se_focal': 'LSTM',
    'resnet_feature': 'ResNet',
    'medium_cnn':'CNN',
    'SeqSleepNet':'SSN'
})

df_results.to_csv("./model_results.csv", index=False)